# Hamshahri Word and 3-gram Index - Without Stop Words

Build a SQLite inverted index from the text documents.It ignores stopwords and then normalize the text by Hazm library. The `inverted_index` table stores  three-word sequences as primary keys in `ngram`. Its `Total` column contains each matching total occurrence count (file1(n)+file2(n)+...). Show top 20 of rows sorted by 'Total' Desc

In [ ]:
import json
import re
import sqlite3
from pathlib import Path

CORPUS_DIR = Path("data/Hamshahri_data")
DATABASE_PATH = Path("data/hamshahri_3gram.db")
WORD_PATTERN = re.compile(r"[^\W_]+", re.UNICODE)


def extract_strings(value):
    """Yield every string value in a nested JSON object or list."""
    if isinstance(value, str):
        yield value
    elif isinstance(value, dict):
        for child in value.values():
            yield from extract_strings(child)
    elif isinstance(value, list):
        for child in value:
            yield from extract_strings(child)


def read_document_text(path):
    """Read a JSON document, or fall back to indexing its raw text."""
    content = path.read_text(encoding="utf-8")
    try:
        document = json.loads(content)
    except json.JSONDecodeError:
        return content
    return " ".join(extract_strings(document))

print(f"Corpus: {CORPUS_DIR}")
print(f"Database: {DATABASE_PATH}")

Corpus: data/Hamshahri_data
Database: data/hamshahri_ngram.db


In [2]:
def build_inverted_index(corpus_dir=CORPUS_DIR, database_path=DATABASE_PATH):
    """Index unigrams, bigrams, and trigrams with per-file occurrence counts."""
    from collections import Counter

    corpus_dir = Path(corpus_dir)
    database_path = Path(database_path)
    documents = sorted(corpus_dir.rglob("*.txt"))
    if not documents:
        raise FileNotFoundError(f"No .txt files found in {corpus_dir.resolve()}")

    database_path.parent.mkdir(parents=True, exist_ok=True)
    index = {}
    for path in documents:
        tokens = [
            token.casefold()
            for token in WORD_PATTERN.findall(read_document_text(path))
        ]
        filename = path.relative_to(corpus_dir).as_posix()
        for gram_size in (1, 2, 3):
            gram_counts = Counter(
                " ".join(tokens[position : position + gram_size])
                for position in range(len(tokens) - gram_size + 1)
            )
            for gram, count in gram_counts.items():
                index.setdefault(gram, []).append((filename, count))

    with sqlite3.connect(database_path) as connection:
        connection.execute("DROP TABLE IF EXISTS inverted_index")
        connection.execute(
            "CREATE TABLE inverted_index ("
            "ngram TEXT PRIMARY KEY NOT NULL, filenames TEXT NOT NULL)"
        )
        connection.executemany(
            "INSERT INTO inverted_index (ngram, filenames) VALUES (?, ?)",
            (
                (
                    gram,
                    " ".join(
                        f"{filename}({count})"
                        for filename, count in sorted(
                            files, key=lambda item: (-item[1], item[0])
                        )
                    ),
                )
                for gram, files in index.items()
            ),
        )

    gram_file_count = sum(len(files) for files in index.values())
    print(
        f"Indexed {len(documents)} documents into {database_path} "
        f"({len(index):,} unique n-grams, {gram_file_count:,} n-gram-file entries)."
    )


build_inverted_index()

Indexed 600 documents into data/hamshahri_ngram.db (281,560 unique n-grams, 417,828 n-gram-file entries).


In [3]:
def find_ngram_files(ngram, database_path=DATABASE_PATH):
    """Return the filename/count string for a one-, two-, or three-word n-gram."""
    tokens = [
        token.casefold()
        for token in WORD_PATTERN.findall(ngram)
    ]
    if not 1 <= len(tokens) <= 3:
        return None

    normalized_ngram = " ".join(tokens)
    with sqlite3.connect(database_path) as connection:
        row = connection.execute(
            "SELECT filenames FROM inverted_index WHERE ngram = ?",
            (normalized_ngram,),
        ).fetchone()
    return row[0] if row else None

In [5]:
search_ngram = input("Enter a word, bigram, or trigram to search: ").strip()
file_occurrences = find_ngram_files(search_ngram)

if file_occurrences:
    print(f"'{search_ngram}' appears in:")
    print(file_occurrences)
else:
    print(f"No files found for '{search_ngram}'.")

'داد' appears in:
HAM2-750404-045.txt(5) HAM2-750406-041.txt(5) HAM2-750402-004.txt(4) HAM2-750402-018.txt(4) HAM2-750404-023.txt(4) HAM2-750407-036.txt(4) HAM2-750409-013.txt(4) HAM2-750402-029.txt(3) HAM2-750403-013.txt(3) HAM2-750403-032.txt(3) HAM2-750403-045.txt(3) HAM2-750404-038.txt(3) HAM2-750404-098.txt(3) HAM2-750406-017.txt(3) HAM2-750402-072.txt(2) HAM2-750402-076.txt(2) HAM2-750402-078.txt(2) HAM2-750402-079.txt(2) HAM2-750403-001.txt(2) HAM2-750404-005.txt(2) HAM2-750404-017.txt(2) HAM2-750405-007.txt(2) HAM2-750405-037.txt(2) HAM2-750405-051.txt(2) HAM2-750406-003.txt(2) HAM2-750407-020.txt(2) HAM2-750407-026.txt(2) HAM2-750407-034.txt(2) HAM2-750407-038.txt(2) HAM2-750407-064.txt(2) HAM2-750409-039.txt(2) HAM2-750409-060.txt(2) HAM2-750402-002.txt(1) HAM2-750402-010.txt(1) HAM2-750402-025.txt(1) HAM2-750402-035.txt(1) HAM2-750402-041.txt(1) HAM2-750402-066.txt(1) HAM2-750402-077.txt(1) HAM2-750403-003.txt(1) HAM2-750403-016.txt(1) HAM2-750403-024.txt(1) HAM2-750403-034.